# Day 03 · Polygons — The Accessible City

**Question.** How much of Chicago is a short walk from an 'L' station that a rider who cannot take
the stairs can use?

**Approach.** Take the 'L' stations and the accessibility rule from Day 01. Measure a half-mile walk
(804.7 m, about ten minutes) along OpenStreetMap's walking network from every station, and again from
vertically accessible stations only, along the same network without stairs. Every 2020 Census block
is placed by its network distance; the reached streets become the plate's polygons. Residents within
a half mile of a station but not of an accessible one live in the **stairs-only zone**.

**Output.** `output/plate.json`, read by the website to draw the plate.

| Section | What happens |
| --- | --- |
| 1. Setup | Imports, paths, run settings |
| 2. Pull | Dated station snapshot (cached); OSM walk network, Census blocks and city limits from `reference/` |
| 3. Validate | Schema, values and consistency checks that stop the run on surprises |
| 4. Transform | Stops → stations (Day 01 rule); walk and step-free networks; distances from every station |
| 5. Analyze | Residents by zone, by line and by station; extra walk; sensitivity; the finding |
| 6. Polygons | Walksheds, stairs-only zone and station catchments, in address units |
| 7. QA view | Quick maps and charts to catch errors before publishing |
| 8. Export | `plate.json` with the data ledger |

Run top to bottom. To refresh the station data, set `REFRESH = True` in section 1. The shared layers
are rebuilt from the repository root: `uv run python -m atlas.census` (blocks) and
`uv run python -m atlas.boundary` (city limits). The OSM extract is a dated Geofabrik file in
`reference/raw/osm/`; the walk network is cut from it and cached on the first run (about ten minutes).

## 1. Setup

In [ ]:
from __future__ import annotations

import json
import logging
import re
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shapely

from atlas import Ledger, snapshot, to_address, walk, write_plate

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s: %(message)s')
logging.getLogger('osmnx').setLevel(logging.WARNING)
log = logging.getLogger('day03')
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 140)

In [ ]:
# Run settings ---------------------------------------------------------------------------------
REFRESH = False  # True = pull a new station snapshot from the Chicago Data Portal

DAY = 3
DATASET_ID = '8pix-ypme'  # CTA - System Information - List of 'L' Stops
DATASET_NAME = "CTA - System Information - List of 'L' Stops"
DATASET_URL = f'https://data.cityofchicago.org/d/{DATASET_ID}'

# Paths: the notebook runs from its own folder (days/day03-polygons/)
DAY_DIR = Path.cwd()
REPO = DAY_DIR.parents[1]
RAW_DIR = DAY_DIR / 'data' / 'raw'
OUT_PATH = DAY_DIR / 'output' / 'plate.json'
OSM_DIR = REPO / walk.OSM_DIR
BLOCKS_CSV = REPO / 'reference' / 'chicago-blocks-2020.csv'
BOUNDARY_RAW = sorted((REPO / 'reference' / 'raw').glob('city_boundary_*.geojson'))[-1]
assert (DAY_DIR / 'day03.ipynb').exists(), f'Run this notebook from its own folder, not {DAY_DIR}'

# Domain constants (as Day 01) -----------------------------------------------------------------
LINES = ('Red', 'Blue', 'Brown', 'Green', 'Orange', 'Purple', 'Pink', 'Yellow')
CHICAGO_BBOX = {'lat': (41.60, 42.10), 'lon': (-87.95, -87.52)}
CTA_PUBLISHED = {'accessible': 110, 'total': 146}  # CTA release of 7 Aug 2026
DAY01_COUNTS = {'total': 144, 'noStepFree': 38}  # Plate I, from the same dataset

# Walk settings --------------------------------------------------------------------------------
WALK_M = walk.HALF_MILE_M  # the half-mile standard for rail station access, 804.7 m
SENSITIVITY = {            # shorter walks, reported in the ledger, not mapped
    'quarter mile': WALK_M / 2,
    'ten minutes at 0.8 m/s': 480.0,  # a slower pace, typical of many mobility-aid users
}
NETWORK_BUFFER_M = 1609.344  # network reaches a mile past the city limits
MAX_SNAP_M = 150             # a station farther than this from any walkway is left out
FRONTAGE_M = 40              # reached streets widened by about one Chicago lot depth (125 ft)
MIN_HOLE_M2 = 20_000         # holes smaller than 2 ha inside a walkshed are filled

# ASAP stations from CTA's ASAP page (https://www.transitchicago.com/accessibility/asap/)
ASAP_PHASE_ONE = [('Montrose', 'Blue'), ('California', 'Blue'), ('Racine', 'Blue'), ('State/Lake', None)]

## 2. Pull

The station list is pulled as in Day 01 (one row per stop) into a dated snapshot. The shared layers
are read from `reference/`: Census blocks (built by `atlas.census` from the 2020 redistricting file and
the ACS 2020–2024 table B18105), the city limits snapshot, and the newest Geofabrik Illinois extract.

In [ ]:
raw, pulled = snapshot(
    DATASET_ID,
    RAW_DIR,
    'l_stops',
    refresh=REFRESH,
    select='map_id,station_descriptive_name,ada,location',
)
log.info('Station snapshot pulled %s: %d rows x %d columns', pulled, *raw.shape)

blocks = pd.read_csv(BLOCKS_CSV, dtype={'geoid': str})
city = shapely.union_all(gpd.read_file(BOUNDARY_RAW).to_crs(walk.CRS).geometry)
boundary_pulled = BOUNDARY_RAW.stem.rsplit('_', 1)[-1]

In [ ]:
# The walk network: city limits plus a mile, cut from the Illinois extract, filtered and cached.
pbf, osm_date = walk.latest_pbf(OSM_DIR)
area = gpd.GeoSeries([city.buffer(NETWORK_BUFFER_M)], crs=walk.CRS).to_crs('EPSG:4326').iloc[0]
xml = walk.extract(pbf, area, OSM_DIR / f'chicago-walk-{osm_date.isoformat()}.osm')
g_all = walk.load(xml)
log.info('OSM extract of %s: %d nodes, %d edges', osm_date, g_all.number_of_nodes(), g_all.number_of_edges())

## 3. Validate

The station checks are Day 01's. The block table must add up to the 2020 count for Chicago, and the
network must contain the elevators and stairs the step-free variant depends on.

In [ ]:
EXPECTED = {'map_id', 'station_descriptive_name', 'ada', 'location.latitude', 'location.longitude'}
missing = EXPECTED - set(raw.columns)
assert not missing, f'Missing columns: {missing}'

stops = raw[list(EXPECTED)].copy()
stops['ada'] = stops['ada'].astype(str).str.lower().str.strip()  # booleans on a fresh pull, text from the CSV
stops['lat'] = pd.to_numeric(stops['location.latitude'], errors='coerce')
stops['lon'] = pd.to_numeric(stops['location.longitude'], errors='coerce')

highways = pd.Series([d.get('highway') for _, _, d in g_all.edges(data=True)]).astype(str)
checks = {
    'no missing station id': stops['map_id'].notna().all(),
    'no missing name': stops['station_descriptive_name'].notna().all(),
    'ADA flag is true/false': stops['ada'].isin({'true', 'false'}).all(),
    'coordinates parse': stops[['lat', 'lon']].notna().all().all(),
    'latitude inside Chicago area': stops['lat'].between(*CHICAGO_BBOX['lat']).all(),
    'longitude inside Chicago area': stops['lon'].between(*CHICAGO_BBOX['lon']).all(),
    'blocks add up to 2020 Chicago': blocks['pop'].sum() == 2_746_388,
    'blocks have coordinates': blocks[['lat', 'lon']].notna().all().all(),
    'network has stairs': highways.str.contains('steps').any(),
    'network has elevators': highways.str.contains('elevator').any(),
}
for name, ok in checks.items():
    log.info('%-32s %s', name, 'ok' if ok else 'FAILED')
assert all(checks.values()), 'Validation failed; see log above'

## 4. Transform

- **Stations.** Day 01's rule, unchanged: a station is vertically accessible only if every one of its
  stops is flagged accessible; lines come from the name's parenthetical.
- **Networks.** OSMnx's `walk` rules applied to the extract, then a step-free copy without
  `highway=steps` and ways tagged `wheelchair=no`. Each keeps only its main connected piece: without
  stairs, subway mezzanines and footbridges become islands that would capture a station's snap.
- **Distances.** One multi-source Dijkstra per network. Each station joins the network at its nearest
  walkway, and the walk from the station point to it counts. Each block's internal point does the same,
  so a block's distance is the walk from its center to the nearest station (and which station that is).

In [ ]:
PAREN = re.compile(r'\((?P<inside>[^)]*)\)\s*$')


def parse_lines(station_name: str) -> list[str]:
    """Return the 'L' lines named in the station's parenthetical, in canonical order."""
    match = PAREN.search(station_name)
    inside = match.group('inside') if match else ''
    return [line for line in LINES if re.search(rf'\b{line}\b', inside)]


def short_name(station_name: str) -> str:
    """Station name without the line parenthetical: 'Clark/Lake (Blue, ...)' -> 'Clark/Lake'."""
    return PAREN.sub('', station_name).strip()


def collapse_to_stations(stops: pd.DataFrame) -> pd.DataFrame:
    """One row per station: vertically accessible only if all stops are; mean stop position."""
    stations = (
        stops.assign(step_free=stops['ada'].eq('true'))
        .groupby('map_id', as_index=False)
        .agg(
            name=('station_descriptive_name', 'first'),
            step_free=('step_free', 'all'),
            lat=('lat', 'mean'),
            lon=('lon', 'mean'),
        )
    )
    stations['name'] = stations['name'].str.replace(r'\s+', ' ', regex=True).str.strip()
    stations['short'] = stations['name'].map(short_name)
    stations['lines'] = stations['name'].map(parse_lines)
    stations['x'], stations['y'] = to_address(stations['lat'], stations['lon'])
    return stations.sort_values('map_id', ignore_index=True)


stations = collapse_to_stations(stops)
assert not (stations['lines'].str.len() == 0).any(), 'A station has no line'
counts = {'total': len(stations), 'noStepFree': int((~stations['step_free']).sum())}
log.info('Stations: %s (Day 01: %s)', counts, DAY01_COUNTS)
assert counts == DAY01_COUNTS, 'Station counts changed since Plate I; explain before publishing'


def utm(lat, lon) -> np.ndarray:
    """(n, 2) meters in the network's projection."""
    pts = gpd.GeoSeries(gpd.points_from_xy(lon, lat), crs='EPSG:4326').to_crs(walk.CRS)
    return np.column_stack([pts.x, pts.y])


st_xy = utm(stations['lat'], stations['lon'])
bl_xy = utm(blocks['lat'], blocks['lon'])
stations['in_city'] = shapely.contains_xy(city, st_xy[:, 0], st_xy[:, 1])

In [ ]:
g_walk = walk.main_component(g_all)
g_free = walk.main_component(walk.step_free(g_all))
E_walk, E_free = walk.Edges.of(g_walk), walk.Edges.of(g_free)
ada = stations['step_free'].to_numpy()
ids = stations['map_id'].tolist()

# distances out to a few miles: far enough to measure the extra walk from the stairs-only zone
d_walk, k_walk, off_walk = walk.reach(E_walk, ids, st_xy, limit=8_000, max_snap=MAX_SNAP_M)
d_free, k_free, off_free = walk.reach(E_free, [i for i, a in zip(ids, ada) if a], st_xy[ada],
                                      limit=12_000, max_snap=MAX_SNAP_M)
stations['snap_m'] = off_walk

# Stations left out of the network must not matter to city residents: outside the city and more than
# a half mile from it, or (O'Hare, inside the terminal) with no resident within a half mile.
left_out = stations[np.isinf(off_walk)].copy()
left_out['to_city_m'] = [city.distance(shapely.Point(p)) for p in st_xy[np.isinf(off_walk)]]
left_out['residents_half_mile'] = [
    int(blocks.loc[np.hypot(*(bl_xy - p).T) <= WALK_M, 'pop'].sum()) for p in st_xy[np.isinf(off_walk)]
]
assert ((left_out['to_city_m'] > WALK_M) | (left_out['residents_half_mile'] == 0)).all(), left_out
log.info('Left out of the network: %s', ', '.join(left_out['short']))
left_out[['map_id', 'short', 'lines', 'in_city', 'to_city_m', 'residents_half_mile']].round()

In [ ]:
blocks['d_walk'], blocks['k_walk'] = walk.point_distance(E_walk, d_walk, k_walk, bl_xy)
blocks['d_free'], blocks['k_free'] = walk.point_distance(E_free, d_free, k_free, bl_xy)
# A step-free route is also a walk. A block can snap to a different walkway on the two networks
# (its nearest one may be a stair), so its walk distance is the shorter of the two.
shorter = blocks['d_free'] < blocks['d_walk']
blocks.loc[shorter, 'k_walk'] = blocks.loc[shorter, 'k_free']
blocks['d_walk'] = blocks[['d_walk', 'd_free']].min(axis=1)


def zone(d_walk: pd.Series, d_free: pd.Series, limit: float) -> pd.Series:
    """'covered' (step-free walk to an accessible station), 'stairs' (walk to a station only), 'beyond'."""
    return pd.Series(np.select([d_free <= limit, d_walk <= limit], ['covered', 'stairs'], 'beyond'),
                     index=d_walk.index)


blocks['zone'] = zone(blocks['d_walk'], blocks['d_free'], WALK_M)
blocks.groupby('zone')['pop'].agg(['size', 'sum'])

## 5. Analyze

In [ ]:
P = int(blocks['pop'].sum())
by_zone = blocks.groupby('zone')[['pop', 'amb', 'amb65']].sum()
near = int(by_zone.loc[['covered', 'stairs'], 'pop'].sum())
stairs = blocks[blocks['zone'] == 'stairs']

# Straight-line comparison: the circle most maps draw
from scipy.spatial import cKDTree
crow_any, _ = cKDTree(st_xy).query(bl_xy)
crow_free, _ = cKDTree(st_xy[ada]).query(bl_xy)

summary = {
    'residents': P,
    'near': near,
    'covered': int(by_zone.loc['covered', 'pop']),
    'stairs': int(by_zone.loc['stairs', 'pop']),
    'nearShare': near / P,
    'coveredShare': by_zone.loc['covered', 'pop'] / P,
    'stairsShare': by_zone.loc['stairs', 'pop'] / P,
    'crowNearShare': blocks.loc[crow_any <= WALK_M, 'pop'].sum() / P,
    'crowCoveredShare': blocks.loc[crow_free <= WALK_M, 'pop'].sum() / P,
    # extra walk from the stairs-only zone to the nearest accessible station (resident-weighted)
    'stairsMedianWalkMi': float(np.interp(0.5, (stairs.sort_values('d_free')['pop'].cumsum() / stairs['pop'].sum()),
                                          stairs.sort_values('d_free')['d_free'])) / 1609.344,
    'stairsBeyondMileShare': stairs.loc[stairs['d_free'] > 1609.344, 'pop'].sum() / stairs['pop'].sum(),
    # ACS estimates allocated to blocks (atlas.census)
    'ambulatory': float(blocks['amb'].sum()),
    'ambulatoryNear': float(by_zone.loc[['covered', 'stairs'], 'amb'].sum()),
    'ambulatoryStairs': float(by_zone.loc['stairs', 'amb']),
    'ambulatoryStairsOver65': float(by_zone.loc['stairs', 'amb65']),
}
log.info('Summary: %s', {k: round(v, 3) if isinstance(v, float) else v for k, v in summary.items()})
pd.Series(summary)

In [ ]:
# Residents near each line: within a half-mile walk of one of the line's stations. Each line gets
# its own runs, so a resident near two lines counts on both. Of those, the stairs-only residents
# (no accessible station of any line within a step-free half mile) and those a line itself serves.
rows = []
for line in LINES:
    on = stations['lines'].map(lambda ls: line in ls).to_numpy()
    d1, k1, _ = walk.reach(E_walk, [i for i, o in zip(ids, on) if o], st_xy[on], limit=WALK_M * 1.5,
                           max_snap=MAX_SNAP_M)
    dl, _ = walk.point_distance(E_walk, d1, k1, bl_xy)
    onf = on & ada
    if onf.any():
        d2, k2, _ = walk.reach(E_free, [i for i, o in zip(ids, onf) if o], st_xy[onf], limit=WALK_M * 1.5,
                               max_snap=MAX_SNAP_M)
        df, _ = walk.point_distance(E_free, d2, k2, bl_xy)
    else:
        df = np.full(len(blocks), np.inf)
    near_line = np.minimum(dl, df) <= WALK_M
    rows.append({
        'line': line,
        'stations': int(on.sum()),
        'nonAccessible': int((on & ~ada).sum()),
        'near': int(blocks.loc[near_line, 'pop'].sum()),
        'lineCovered': int(blocks.loc[near_line & (df <= WALK_M), 'pop'].sum()),
        'stairs': int(blocks.loc[near_line & (blocks['zone'] == 'stairs'), 'pop'].sum()),
    })
by_line = pd.DataFrame(rows)
by_line.sort_values('stairs', ascending=False)

In [ ]:
# Residents by station: the catchment of each station on the walk network (the nearest station),
# within a half mile; and the stairs-only residents in it.
near_blocks = blocks[blocks['zone'] != 'beyond']
by_station = (
    near_blocks.groupby('k_walk')
    .agg(residents=('pop', 'sum'), stairs=('pop', lambda s: int(s[near_blocks.loc[s.index, 'zone'] == 'stairs'].sum())))
    .reindex(stations['map_id'], fill_value=0)
)
stations = stations.join(by_station, on='map_id')
stations[['residents', 'stairs']] = stations[['residents', 'stairs']].fillna(0).astype(int)
assert stations['stairs'].sum() == summary['stairs']
top_stairs = stations.sort_values('stairs', ascending=False).head(12)
top_stairs[['map_id', 'short', 'lines', 'step_free', 'residents', 'stairs']]

In [ ]:
# Sensitivity: shorter walks, and two changes to the station list.
sens = {}
for label, limit in SENSITIVITY.items():
    z = zone(blocks['d_walk'], blocks['d_free'], limit)
    sens[label] = {'nearShare': blocks.loc[z != 'beyond', 'pop'].sum() / P,
                   'coveredShare': blocks.loc[z == 'covered', 'pop'].sum() / P}


def with_accessible(extra: list[str]) -> dict:
    """Residents in the stairs-only zone if these stations were vertically accessible too."""
    keep = ada | stations['map_id'].isin(extra).to_numpy()
    d, k, _ = walk.reach(E_free, [i for i, a in zip(ids, keep) if a], st_xy[keep], limit=WALK_M * 1.5,
                         max_snap=MAX_SNAP_M)
    df, _ = walk.point_distance(E_free, d, k, bl_xy)
    z = zone(blocks['d_walk'], pd.Series(df, index=blocks.index), WALK_M)
    return {'stairs': int(blocks.loc[z == 'stairs', 'pop'].sum()),
            'covered': int(blocks.loc[z == 'covered', 'pop'].sum())}


def station_id(short: str, line: str | None) -> str:
    """The non-accessible station with this name (and line); exactly one must match."""
    m = stations[(stations['short'] == short) & ~stations['step_free']]
    if line:
        m = m[m['lines'].map(lambda ls: line in ls)]
    assert len(m) == 1, (short, line, m[['map_id', 'name']].values.tolist())
    return m['map_id'].iloc[0]


# Austin (Green) reopened accessible in Aug 2026 but is still flagged non-accessible (Day 01 caveat)
austin_green = station_id('Austin', 'Green')
sens['Austin (Green) as accessible'] = with_accessible([austin_green])
# ASAP Phase One stations not yet accessible in the data
phase_one = [station_id(s, l) for s, l in ASAP_PHASE_ONE]
sens['ASAP Phase One done'] = with_accessible(phase_one)
log.info('Phase One stations: %s', stations.set_index('map_id').loc[phase_one, 'name'].tolist())
pd.DataFrame(sens).T

In [ ]:
# The finding printed on the plate, generated from the data so it can never drift from the numbers.
pct = lambda v: f'{100 * v:.0f}%'
thousands = lambda n: f'{round(n, -3):,.0f}'
FINDING = (
    f'{pct(summary["nearShare"])} of Chicagoans live within a half-mile walk of an \u2018L\u2019 station; '
    f'{pct(summary["coveredShare"])} live that close to one they can use without stairs. '
    f'About {thousands(summary["stairs"])} live in between.'
)
print(FINDING)

## 6. Polygons

The reached part of every edge (cut where the half mile runs out, and where two stations' walks meet)
is widened by `FRONTAGE_M`, merged, and clipped to the city limits. The stairs-only zone is the walkshed
of all stations minus the step-free walkshed of accessible stations. Each station's catchment is the
part of the walkshed nearest to it. Rings are converted to address units and simplified (15 units,
about 100 ft; catchments 25).

In [ ]:
lines_walk = walk.reached(E_walk, d_walk, k_walk, WALK_M)
lines_free = walk.reached(E_free, d_free, k_free, WALK_M)
shed_walk = walk.shed(lines_walk.geometry, FRONTAGE_M, MIN_HOLE_M2).intersection(city)
shed_free = walk.shed(lines_free.geometry, FRONTAGE_M, MIN_HOLE_M2).intersection(city)
shed_stairs = shed_walk.difference(shed_free)
catchments = {k: walk.shed(grp.geometry, FRONTAGE_M, MIN_HOLE_M2).intersection(city)
              for k, grp in lines_walk.groupby('key')}

SQ_MI = 2_589_988.11
area = {
    'citySqMi': city.area / SQ_MI,
    'nearSqMi': shed_walk.area / SQ_MI,
    'coveredSqMi': shed_free.area / SQ_MI,
    'stairsSqMi': shed_stairs.area / SQ_MI,
}
# circles of a half mile around every station, for the network-vs-circle comparison
circles = shapely.union_all(shapely.buffer(shapely.points(st_xy), WALK_M)).intersection(city)
area['circleSqMi'] = circles.area / SQ_MI
log.info('Areas (sq mi): %s', {k: round(v, 1) for k, v in area.items()})

# The largest piece of the stairs-only zone by residents frames the inset.
parts = gpd.GeoDataFrame(geometry=list(getattr(shed_stairs, 'geoms', [shed_stairs])), crs=walk.CRS)
pts = gpd.GeoDataFrame(stairs[['pop']], geometry=gpd.points_from_xy(*bl_xy[stairs.index].T), crs=walk.CRS)
hits = gpd.sjoin(pts, parts, predicate='within', how='left')
part_pop = hits.groupby('index_right')['pop'].sum().sort_values(ascending=False)
log.info('Stairs-only zone: %d pieces; covered by block points: %.1f%% of its residents',
         len(parts), 100 * part_pop.sum() / stairs['pop'].sum())
largest = parts.geometry.iloc[int(part_pop.index[0])]
inset_stations = stations[[largest.distance(shapely.Point(p)) <= WALK_M for p in st_xy]]
log.info('Largest piece: %d residents; stations near it: %s', part_pop.iloc[0], ', '.join(inset_stations['short']))

In [ ]:
rings = {
    'near': walk.to_rings(shed_walk),
    'covered': walk.to_rings(shed_free),
    'stairs': walk.to_rings(shed_stairs),
}
catch_rings = {k: walk.to_rings(v, tolerance=25) for k, v in catchments.items() if not v.is_empty}
lb = gpd.GeoSeries([largest], crs=walk.CRS).to_crs('EPSG:4326').total_bounds  # lon0, lat0, lon1, lat1
ix, iy = to_address([lb[1], lb[3]], [lb[0], lb[2]])
inset_box = {'west': float(ix[0]), 'east': float(ix[1]), 'south': float(iy[0]), 'north': float(iy[1])}
log.info('Vertices: %s; catchments: %d; inset box %s',
         {k: sum(len(r) for p in v for r in p) for k, v in rings.items()}, len(catch_rings),
         {k: round(v) for k, v in inset_box.items()})

## 7. QA view

A plain check, not the final design: the three zones with stations on top, and residents near each
line. Look for walksheds that stop at a barrier that is not there (missing crossings), stations off
the network, or a line with an implausible count.

In [ ]:
fig, (ax_map, ax_bar) = plt.subplots(1, 2, figsize=(12, 8), gridspec_kw={'width_ratios': [1, 1]})
gpd.GeoSeries([city], crs=walk.CRS).plot(ax=ax_map, color='0.92')
gpd.GeoSeries([shed_free], crs=walk.CRS).plot(ax=ax_map, color='0.55')
gpd.GeoSeries([shed_stairs], crs=walk.CRS).plot(ax=ax_map, facecolor='none', hatch='////', edgecolor='0.15', linewidth=0)
ax_map.scatter(st_xy[ada, 0], st_xy[ada, 1], s=8, color='0.1', zorder=3)
ax_map.scatter(st_xy[~ada, 0], st_xy[~ada, 1], s=14, facecolor='white', edgecolor='0.1', zorder=3)
gpd.GeoSeries([largest], crs=walk.CRS).boundary.plot(ax=ax_map, color='red', linewidth=0.8)
minx, miny, maxx, maxy = city.bounds
ax_map.set(xlim=(minx, maxx), ylim=(miny, maxy), title='Step-free walkshed (gray), stairs-only zone (hatched)')
ax_map.set_axis_off()

order = by_line.sort_values('stairs')
ax_bar.barh(order['line'], order['near'], color='0.85', label='residents near the line')
ax_bar.barh(order['line'], order['stairs'], color='0.15', label='of them, stairs-only')
ax_bar.set(title='Residents within a half-mile walk, by line', xlabel='residents')
ax_bar.legend(loc='lower right', frameon=False)
for side in ('top', 'right'):
    ax_bar.spines[side].set_visible(False)
fig.tight_layout()
plt.show()

# network vs straight line: the walk to the nearest station over the crow-fly distance (half-mile to
# two-mile blocks); a grid city should sit near 4/pi = 1.27, much higher means broken connectivity
ratio = (blocks['d_walk'] / crow_any)[(crow_any > 400) & (crow_any < 3200)]
log.info('Detour ratio quantiles: %s', ratio.quantile([0.1, 0.5, 0.9, 0.99]).round(2).to_dict())
assert 1.15 < ratio.median() < 1.5, 'Walk distances do not look like a street grid'

## 8. Export

`plate.json` carries the data ledger, the finding, the summary and tables, the polygons in address
units and the inset box. Keys are camelCase because the website reads them directly.

In [ ]:
ledger = Ledger(
    source=DATASET_NAME,
    dataset_id=DATASET_ID,
    url=DATASET_URL,
    pulled=pulled,
    units=f'{P:,} residents (2020 Census) in {len(blocks):,} blocks; {counts["total"]} stations (map_id)',
    method=[
        f'Half-mile walk ({WALK_M:.0f} m) on the OpenStreetMap walk network, extract of {osm_date}',
        'Step-free: the same network without stairs and ways tagged wheelchair=no',
        'Blocks placed by the walk from their internal point to the nearest station',
        'Stations: Day 01 rule, accessible only if every platform is flagged accessible',
    ],
    caveats=[
        'OSM does not record every missing curb ramp or broken sidewalk; the step-free walk is an upper bound',
        'ADA flag lags CTA: Austin (Green), reopened accessible Aug 2026, is still flagged non-accessible',
        'Ambulatory difficulty: ACS 2020-2024 tract estimates spread over blocks by population',
    ],
)

r1 = lambda v: round(float(v), 1)
payload = {
    'source': {'name': DATASET_NAME, 'id': DATASET_ID, 'url': DATASET_URL},
    'osm': {'extract': osm_date.isoformat(), 'source': 'Geofabrik, Illinois',
            'url': 'https://download.geofabrik.de/north-america/us/illinois.html'},
    'census': {'blocks': len(blocks), 'boundaryPulled': boundary_pulled},
    'walkM': round(WALK_M, 1),
    'summary': {**summary, **area},
    'sensitivity': sens,
    'leftOut': left_out['short'].tolist(),
    'phaseOne': stations.set_index('map_id').loc[phase_one, 'short'].tolist(),
    'byLine': by_line.to_dict(orient='records'),
    'stations': [
        {
            'id': r.map_id, 'name': r.name, 'short': r.short, 'lines': r.lines,
            'ada': bool(r.step_free), 'lat': round(r.lat, 6), 'lon': round(r.lon, 6),
            'x': r1(r.x), 'y': r1(r.y), 'residents': int(r.residents), 'stairs': int(r.stairs),
            'onNetwork': bool(np.isfinite(r.snap_m)),
        }
        for r in stations.itertuples()
    ],
    'polygons': rings,
    'catchments': catch_rings,
    'inset': {'box': inset_box, 'residents': int(part_pop.iloc[0]), 'stations': inset_stations['map_id'].tolist()},
}

path = write_plate(OUT_PATH, day=DAY, ledger=ledger, finding=FINDING, payload=payload)

# Read back and confirm the contract the website depends on.
written = json.loads(path.read_text())
assert {'asOf', 'finding', 'ledger', 'summary', 'byLine', 'stations', 'polygons', 'catchments', 'inset'} <= written.keys()
assert len(written['stations']) == counts['total']
log.info('Wrote %s (%.1f KB)', path.relative_to(DAY_DIR), path.stat().st_size / 1024)

## Notes for the plate

- Finding: printed in section 5; it is generated from the data.
- Caveats for the ledger: the step-free network only knows what OSM records (stairs, `wheelchair=no`),
  not every missing curb ramp, so the step-free walkshed is an upper bound; the portal's ADA flag still
  marks Austin (Green) non-accessible (sensitivity in section 5); ambulatory difficulty is an estimate.
- Next: the website reads `output/plate.json`; rebuild the site and export the PNG.